# Home Credit Data Engineering

## Dataset Ingestion and Structural Audit

This notebook engineers the Home Credit dataset as a distributed multi-table credit-risk benchmark/reference layer.

The first stage inventories the available tables, schemas, row counts, key fields, and data types before designing the aggregation and feature-engineering pipeline.

The Home Credit dataset is used as a separate reference engineering/modeling dataset and is not merged at the customer level with the Nigerian BNPL dataset.

In [0]:
from pyspark.sql import functions as F

# ============================================================
# Home Credit source location
# ============================================================

BASE_PATH = "/Volumes/workspace/default/home_credit_raw"

# Required Home Credit tables
TABLE_FILES = {
    "application_train": f"{BASE_PATH}/application_train.csv",
    "application_test": f"{BASE_PATH}/application_test.csv",
    "bureau": f"{BASE_PATH}/bureau.csv",
    "bureau_balance": f"{BASE_PATH}/bureau_balance.csv",
    "previous_application": f"{BASE_PATH}/previous_application.csv",
    "installments_payments": f"{BASE_PATH}/installments_payments.csv",
    "POS_CASH_balance": f"{BASE_PATH}/POS_CASH_balance.csv",
    "credit_card_balance": f"{BASE_PATH}/credit_card_balance.csv",
}

# ============================================================
# Read all tables into Spark DataFrames
# ============================================================

home_credit = {}

for table_name, file_path in TABLE_FILES.items():
    print(f"\nReading: {table_name}")
    
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("mode", "PERMISSIVE")
        .csv(file_path)
    )
    
    home_credit[table_name] = df
    
    print(f"Columns: {len(df.columns)}")
    print("Schema:")
    df.printSchema()


# ============================================================
# Row-count and column-count inventory
# ============================================================

print("\n" + "=" * 80)
print("HOME CREDIT TABLE INVENTORY")
print("=" * 80)

inventory_rows = []

for table_name, df in home_credit.items():
    row_count = df.count()
    column_count = len(df.columns)
    
    inventory_rows.append(
        (table_name, row_count, column_count)
    )

inventory_df = spark.createDataFrame(
    inventory_rows,
    ["table_name", "row_count", "column_count"]
)

display(
    inventory_df.orderBy(F.desc("row_count"))
)


# ============================================================
# Column inventory
# ============================================================

print("\n" + "=" * 80)
print("COLUMN INVENTORY")
print("=" * 80)

column_inventory_rows = []

for table_name, df in home_credit.items():
    for field in df.schema.fields:
        column_inventory_rows.append(
            (
                table_name,
                field.name,
                field.dataType.simpleString(),
                field.nullable
            )
        )

column_inventory_df = spark.createDataFrame(
    column_inventory_rows,
    ["table_name", "column_name", "data_type", "nullable"]
)

display(
    column_inventory_df.orderBy("table_name", "column_name")
)


# ============================================================
# Key-field availability audit
# ============================================================

print("\n" + "=" * 80)
print("KEY FIELD AUDIT")
print("=" * 80)

expected_keys = {
    "application_train": ["SK_ID_CURR", "TARGET"],
    "application_test": ["SK_ID_CURR"],
    "bureau": ["SK_ID_CURR", "SK_ID_BUREAU"],
    "bureau_balance": ["SK_ID_BUREAU"],
    "previous_application": ["SK_ID_CURR", "SK_ID_PREV"],
    "installments_payments": ["SK_ID_CURR", "SK_ID_PREV"],
    "POS_CASH_balance": ["SK_ID_CURR", "SK_ID_PREV"],
    "credit_card_balance": ["SK_ID_CURR", "SK_ID_PREV"],
}

key_audit_rows = []

for table_name, expected_columns in expected_keys.items():
    df = home_credit[table_name]
    
    for col_name in expected_columns:
        exists = col_name in df.columns
        
        if exists:
            null_count = df.filter(F.col(col_name).isNull()).count()
        else:
            null_count = None
        
        key_audit_rows.append(
            (
                table_name,
                col_name,
                exists,
                null_count
            )
        )

key_audit_df = spark.createDataFrame(
    key_audit_rows,
    ["table_name", "key_column", "exists", "null_count"]
)

display(key_audit_df)


# ============================================================
# Application target distribution
# ============================================================

print("\n" + "=" * 80)
print("APPLICATION TRAIN TARGET DISTRIBUTION")
print("=" * 80)

application_train = home_credit["application_train"]

target_distribution = (
    application_train
    .groupBy("TARGET")
    .count()
    .withColumn(
        "percentage",
        F.round(
            F.col("count") / application_train.count() * 100,
            3
        )
    )
    .orderBy("TARGET")
)

display(target_distribution)


# ============================================================
# Date / temporal field inventory
# ============================================================

print("\n" + "=" * 80)
print("TEMPORAL FIELD INVENTORY")
print("=" * 80)

date_keywords = [
    "DAYS",
    "MONTHS",
    "DATE",
    "YEAR",
    "WEEK",
    "HOUR"
]

temporal_rows = []

for table_name, df in home_credit.items():
    for col_name in df.columns:
        if any(keyword in col_name.upper() for keyword in date_keywords):
            temporal_rows.append(
                (table_name, col_name)
            )

temporal_df = spark.createDataFrame(
    temporal_rows,
    ["table_name", "temporal_column"]
)

display(
    temporal_df.orderBy("table_name", "temporal_column")
)


# ============================================================
# Final confirmation
# ============================================================

print("\n" + "=" * 80)
print("AUDIT COMPLETE")
print("=" * 80)

print(f"Tables loaded: {len(home_credit)}")
print("Home Credit source path:")
print(BASE_PATH)
print("\nNext step: use the actual schemas and table relationships to design")
print("the distributed aggregation pipeline before joining historical tables.")


Reading: application_train
Columns: 122
Schema:
root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- TARGET: integer (nullable = true)
 |-- NAME_CONTRACT_TYPE: string (nullable = true)
 |-- CODE_GENDER: string (nullable = true)
 |-- FLAG_OWN_CAR: string (nullable = true)
 |-- FLAG_OWN_REALTY: string (nullable = true)
 |-- CNT_CHILDREN: integer (nullable = true)
 |-- AMT_INCOME_TOTAL: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- AMT_GOODS_PRICE: double (nullable = true)
 |-- NAME_TYPE_SUITE: string (nullable = true)
 |-- NAME_INCOME_TYPE: string (nullable = true)
 |-- NAME_EDUCATION_TYPE: string (nullable = true)
 |-- NAME_FAMILY_STATUS: string (nullable = true)
 |-- NAME_HOUSING_TYPE: string (nullable = true)
 |-- REGION_POPULATION_RELATIVE: double (nullable = true)
 |-- DAYS_BIRTH: integer (nullable = true)
 |-- DAYS_EMPLOYED: integer (nullable = true)
 |-- DAYS_REGISTRATION: double (nullable = true)
 |-- DAYS_ID_P

table_name,row_count,column_count
bureau_balance,27299925,3
installments_payments,13605401,8
POS_CASH_balance,10001358,8
credit_card_balance,3840312,23
bureau,1716428,17
previous_application,1670214,37
application_train,307511,122
application_test,48744,121



COLUMN INVENTORY


table_name,column_name,data_type,nullable
POS_CASH_balance,CNT_INSTALMENT,double,true
POS_CASH_balance,CNT_INSTALMENT_FUTURE,double,true
POS_CASH_balance,MONTHS_BALANCE,int,true
POS_CASH_balance,NAME_CONTRACT_STATUS,string,true
POS_CASH_balance,SK_DPD,int,true
POS_CASH_balance,SK_DPD_DEF,int,true
POS_CASH_balance,SK_ID_CURR,int,true
POS_CASH_balance,SK_ID_PREV,int,true
application_test,AMT_ANNUITY,double,true
application_test,AMT_CREDIT,double,true



KEY FIELD AUDIT


...


APPLICATION TRAIN TARGET DISTRIBUTION


TARGET,count,percentage
0,282686,91.927
1,24825,8.073



TEMPORAL FIELD INVENTORY


table_name,temporal_column
POS_CASH_balance,MONTHS_BALANCE
application_test,AMT_REQ_CREDIT_BUREAU_HOUR
application_test,AMT_REQ_CREDIT_BUREAU_WEEK
application_test,AMT_REQ_CREDIT_BUREAU_YEAR
application_test,DAYS_BIRTH
application_test,DAYS_EMPLOYED
application_test,DAYS_ID_PUBLISH
application_test,DAYS_LAST_PHONE_CHANGE
application_test,DAYS_REGISTRATION
application_test,HOUR_APPR_PROCESS_START



AUDIT COMPLETE
Tables loaded: 8
Home Credit source path:
/Volumes/workspace/default/home_credit_raw

Next step: use the actual schemas and table relationships to design
the distributed aggregation pipeline before joining historical tables.


## Bronze Layer and Relational Integrity

The raw Home Credit CSV files are ingested into distributed Spark DataFrames and persisted as Delta-format Bronze datasets.

Because the source contains multiple one-to-many relationships, historical tables are not directly joined to the application table. Instead, child-level histories will first be aggregated to their natural parent keys before being joined upward.

The main relationships are:

- application_train/application_test → SK_ID_CURR
- bureau → SK_ID_CURR, SK_ID_BUREAU
- bureau_balance → SK_ID_BUREAU
- previous_application → SK_ID_CURR, SK_ID_PREV
- installments_payments → SK_ID_PREV
- POS_CASH_balance → SK_ID_PREV
- credit_card_balance → SK_ID_PREV

This hierarchical aggregation prevents row multiplication and preserves the application-level modelling grain.

In [0]:
from pyspark.sql import functions as F

# ============================================================
# Home Credit Bronze Layer
# ============================================================

BASE_PATH = "/Volumes/workspace/default/home_credit_raw"
BRONZE_PATH = f"{BASE_PATH}/bronze"

# The DataFrames loaded during the previous audit
# are stored in the home_credit dictionary.

# ------------------------------------------------------------
# Persist each source table as Delta
# ------------------------------------------------------------

bronze_paths = {}

for table_name, df in home_credit.items():

    output_path = f"{BRONZE_PATH}/{table_name}"

    print(f"Writing Bronze Delta: {table_name}")

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(output_path)
    )

    bronze_paths[table_name] = output_path

print("\nBronze layer creation complete.")


# ============================================================
# Reload Bronze datasets
# ============================================================

bronze = {}

for table_name, path in bronze_paths.items():
    bronze[table_name] = spark.read.format("delta").load(path)

print(f"Bronze tables available: {len(bronze)}")


# ============================================================
# Bronze row-count verification
# ============================================================

bronze_inventory = []

for table_name, df in bronze.items():

    bronze_inventory.append(
        (
            table_name,
            df.count(),
            len(df.columns)
        )
    )

bronze_inventory_df = spark.createDataFrame(
    bronze_inventory,
    ["table_name", "row_count", "column_count"]
)

print("\n" + "=" * 80)
print("BRONZE LAYER INVENTORY")
print("=" * 80)

display(
    bronze_inventory_df.orderBy(F.desc("row_count"))
)


# ============================================================
# Key uniqueness / cardinality checks
# ============================================================

print("\n" + "=" * 80)
print("KEY CARDINALITY AUDIT")
print("=" * 80)

key_checks = [
    ("application_train", "SK_ID_CURR"),
    ("application_test", "SK_ID_CURR"),
    ("bureau", "SK_ID_BUREAU"),
    ("bureau_balance", "SK_ID_BUREAU"),
    ("previous_application", "SK_ID_PREV"),
    ("installments_payments", "SK_ID_PREV"),
    ("POS_CASH_balance", "SK_ID_PREV"),
    ("credit_card_balance", "SK_ID_PREV"),
]

cardinality_rows = []

for table_name, key_column in key_checks:

    df = bronze[table_name]

    total_rows = df.count()
    distinct_keys = (
        df
        .select(key_column)
        .distinct()
        .count()
    )

    cardinality_rows.append(
        (
            table_name,
            key_column,
            total_rows,
            distinct_keys,
            total_rows - distinct_keys
        )
    )

cardinality_df = spark.createDataFrame(
    cardinality_rows,
    [
        "table_name",
        "key_column",
        "row_count",
        "distinct_key_count",
        "duplicate_key_rows"
    ]
)

display(cardinality_df)


# ============================================================
# Parent-child relationship coverage
# ============================================================

print("\n" + "=" * 80)
print("PARENT-CHILD RELATIONSHIP COVERAGE")
print("=" * 80)


def relationship_coverage(
    child_df,
    child_key,
    parent_df,
    parent_key,
    relationship_name
):
    child_keys = (
        child_df
        .select(child_key)
        .where(F.col(child_key).isNotNull())
        .distinct()
    )

    parent_keys = (
        parent_df
        .select(parent_key)
        .where(F.col(parent_key).isNotNull())
        .distinct()
    )

    unmatched = (
        child_keys
        .join(
            parent_keys,
            child_keys[child_key] == parent_keys[parent_key],
            "left_anti"
        )
        .count()
    )

    total_child_keys = child_keys.count()

    matched = total_child_keys - unmatched

    coverage = (
        matched / total_child_keys * 100
        if total_child_keys > 0
        else 0
    )

    return (
        relationship_name,
        total_child_keys,
        matched,
        unmatched,
        round(coverage, 3)
    )


relationship_results = []

relationship_results.append(
    relationship_coverage(
        bronze["bureau_balance"],
        "SK_ID_BUREAU",
        bronze["bureau"],
        "SK_ID_BUREAU",
        "bureau_balance → bureau"
    )
)

relationship_results.append(
    relationship_coverage(
        bronze["bureau"],
        "SK_ID_CURR",
        bronze["application_train"],
        "SK_ID_CURR",
        "bureau → application_train"
    )
)

relationship_results.append(
    relationship_coverage(
        bronze["previous_application"],
        "SK_ID_CURR",
        bronze["application_train"],
        "SK_ID_CURR",
        "previous_application → application_train"
    )
)

relationship_results.append(
    relationship_coverage(
        bronze["installments_payments"],
        "SK_ID_PREV",
        bronze["previous_application"],
        "SK_ID_PREV",
        "installments_payments → previous_application"
    )
)

relationship_results.append(
    relationship_coverage(
        bronze["POS_CASH_balance"],
        "SK_ID_PREV",
        bronze["previous_application"],
        "SK_ID_PREV",
        "POS_CASH_balance → previous_application"
    )
)

relationship_results.append(
    relationship_coverage(
        bronze["credit_card_balance"],
        "SK_ID_PREV",
        bronze["previous_application"],
        "SK_ID_PREV",
        "credit_card_balance → previous_application"
    )
)

relationship_df = spark.createDataFrame(
    relationship_results,
    [
        "relationship",
        "child_distinct_keys",
        "matched_keys",
        "unmatched_keys",
        "coverage_pct"
    ]
)

display(relationship_df)


# ============================================================
# Historical depth by customer/application
# ============================================================

print("\n" + "=" * 80)
print("HISTORICAL DEPTH")
print("=" * 80)

bureau_depth = (
    bronze["bureau"]
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("*").alias("bureau_records")
    )
)

previous_depth = (
    bronze["previous_application"]
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("*").alias("previous_application_records")
    )
)

historical_depth = (
    bronze["application_train"]
    .select("SK_ID_CURR")
    .join(bureau_depth, "SK_ID_CURR", "left")
    .join(previous_depth, "SK_ID_CURR", "left")
    .fillna(0)
)

display(
    historical_depth.select(
        "SK_ID_CURR",
        "bureau_records",
        "previous_application_records"
    ).limit(20)
)


# ============================================================
# Final architecture confirmation
# ============================================================

print("\n" + "=" * 80)
print("BRONZE ENGINEERING AUDIT COMPLETE")
print("=" * 80)

print("Source CSVs → Bronze Delta: COMPLETE")
print("Application grain: SK_ID_CURR")
print("Bureau parent grain: SK_ID_BUREAU")
print("Previous application grain: SK_ID_PREV")
print("Historical tables will be aggregated before joining.")
print("\nNext engineering stage:")
print("bureau_balance → bureau")
print("installments/POS/credit_card → previous_application")
print("previous_application + bureau → application/customer level")

Writing Bronze Delta: application_train
Writing Bronze Delta: application_test
Writing Bronze Delta: bureau
Writing Bronze Delta: bureau_balance
Writing Bronze Delta: previous_application
Writing Bronze Delta: installments_payments
Writing Bronze Delta: POS_CASH_balance
Writing Bronze Delta: credit_card_balance

Bronze layer creation complete.
Bronze tables available: 8

BRONZE LAYER INVENTORY


table_name,row_count,column_count
bureau_balance,27299925,3
installments_payments,13605401,8
POS_CASH_balance,10001358,8
credit_card_balance,3840312,23
bureau,1716428,17
previous_application,1670214,37
application_train,307511,122
application_test,48744,121



KEY CARDINALITY AUDIT


table_name,key_column,row_count,distinct_key_count,duplicate_key_rows
application_train,SK_ID_CURR,307511,307511,0
application_test,SK_ID_CURR,48744,48744,0
bureau,SK_ID_BUREAU,1716428,1716428,0
bureau_balance,SK_ID_BUREAU,27299925,817395,26482530
previous_application,SK_ID_PREV,1670214,1670214,0
installments_payments,SK_ID_PREV,13605401,997752,12607649
POS_CASH_balance,SK_ID_PREV,10001358,936325,9065033
credit_card_balance,SK_ID_PREV,3840312,104307,3736005



PARENT-CHILD RELATIONSHIP COVERAGE


relationship,child_distinct_keys,matched_keys,unmatched_keys,coverage_pct
bureau_balance → bureau,817395,774354,43041,94.734
bureau → application_train,305811,263491,42320,86.161
previous_application → application_train,338857,291057,47800,85.894
installments_payments → previous_application,997752,958905,38847,96.107
POS_CASH_balance → previous_application,936325,898903,37422,96.003
credit_card_balance → previous_application,104307,92935,11372,89.098



HISTORICAL DEPTH


SK_ID_CURR,bureau_records,previous_application_records
387559,0,5
387560,5,1
387561,3,3
387562,2,2
387563,18,3
387564,8,3
387566,1,4
387567,5,1
387568,6,5
387569,13,7



BRONZE ENGINEERING AUDIT COMPLETE
Source CSVs → Bronze Delta: COMPLETE
Application grain: SK_ID_CURR
Bureau parent grain: SK_ID_BUREAU
Previous application grain: SK_ID_PREV
Historical tables will be aggregated before joining.

Next engineering stage:
bureau_balance → bureau
installments/POS/credit_card → previous_application
previous_application + bureau → application/customer level


## Hierarchical Historical Aggregation

The Home Credit dataset contains several one-to-many historical relationships.

To preserve the application-level modelling grain, each historical table is first aggregated at its natural parent key:

1. bureau_balance is aggregated to SK_ID_BUREAU
2. bureau is then aggregated to SK_ID_CURR
3. installments_payments is aggregated to SK_ID_PREV
4. POS_CASH_balance is aggregated to SK_ID_PREV
5. credit_card_balance is aggregated to SK_ID_PREV
6. previous_application is enriched with these aggregates and then aggregated to SK_ID_CURR
7. The resulting historical features are joined to the application-level dataset

This prevents row multiplication and converts high-volume transactional histories into compact application-level risk features.

The target variable TARGET is never used during historical aggregation.

In [0]:
from pyspark.sql import functions as F


# ============================================================
# Use Bronze Delta tables
# ============================================================

bronze = {}

for table_name in TABLE_FILES.keys():
    bronze[table_name] = (
        spark.read
        .format("delta")
        .load(f"{BRONZE_PATH}/{table_name}")
    )


# ============================================================
# Bureau balance → Bureau account level
# ============================================================

print("=" * 80)
print("AGGREGATING BUREAU BALANCE")
print("=" * 80)

bureau_balance_agg = (
    bronze["bureau_balance"]
    .groupBy("SK_ID_BUREAU")
    .agg(
        F.count("*").alias("bureau_balance_months"),
        
        F.sum(
            F.when(F.col("STATUS").isin("1", "2", "3", "4", "5"), 1)
            .otherwise(0)
        ).alias("bureau_delinquent_months"),
        
        F.sum(
            F.when(F.col("STATUS") == "0", 1)
            .otherwise(0)
        ).alias("bureau_current_months"),
        
        F.sum(
            F.when(F.col("STATUS") == "C", 1)
            .otherwise(0)
        ).alias("bureau_closed_months"),
        
        F.sum(
            F.when(F.col("STATUS") == "X", 1)
            .otherwise(0)
        ).alias("bureau_unknown_months")
    )
)

print(
    f"Bureau balance aggregate rows: "
    f"{bureau_balance_agg.count():,}"
)

display(bureau_balance_agg.limit(10))


# ============================================================
# Bureau account + bureau balance → customer/application level
# ============================================================

print("=" * 80)
print("AGGREGATING BUREAU HISTORY TO APPLICATION LEVEL")
print("=" * 80)

bureau_enriched = (
    bronze["bureau"]
    .join(
        bureau_balance_agg,
        on="SK_ID_BUREAU",
        how="left"
    )
)

bureau_agg = (
    bureau_enriched
    .groupBy("SK_ID_CURR")
    .agg(
        # Number of historical bureau accounts
        F.count("*").alias("bureau_account_count"),
        
        F.sum(
            F.when(F.col("CREDIT_ACTIVE") == "Active", 1)
            .otherwise(0)
        ).alias("bureau_active_count"),
        
        F.sum(
            F.when(F.col("CREDIT_ACTIVE") == "Closed", 1)
            .otherwise(0)
        ).alias("bureau_closed_count"),
        
        # Credit exposure
        F.sum("AMT_CREDIT_SUM").alias("bureau_total_credit"),
        F.sum("AMT_CREDIT_SUM_DEBT").alias("bureau_total_debt"),
        F.sum("AMT_CREDIT_SUM_OVERDUE").alias("bureau_total_overdue"),
        
        F.max("AMT_CREDIT_SUM").alias("bureau_max_credit"),
        F.max("AMT_CREDIT_SUM_DEBT").alias("bureau_max_debt"),
        F.max("AMT_CREDIT_SUM_OVERDUE").alias("bureau_max_overdue"),
        
        # Delinquency
        F.max("CREDIT_DAY_OVERDUE").alias("bureau_max_days_overdue"),
        
        F.sum(
            F.when(F.col("CREDIT_DAY_OVERDUE") > 0, 1)
            .otherwise(0)
        ).alias("bureau_overdue_account_count"),
        
        # Historical timing
        F.min("DAYS_CREDIT").alias("bureau_oldest_credit_days"),
        F.max("DAYS_CREDIT").alias("bureau_most_recent_credit_days"),
        
        # Credit characteristics
        F.countDistinct("CREDIT_TYPE").alias("bureau_credit_type_count"),
        F.countDistinct("CREDIT_CURRENCY").alias("bureau_currency_count"),
        
        # Prolongations
        F.sum("CNT_CREDIT_PROLONG").alias("bureau_total_prolongations"),
        
        # Bureau monthly-history behaviour
        F.sum("bureau_balance_months").alias("bureau_balance_months"),
        F.sum("bureau_delinquent_months").alias("bureau_delinquent_months"),
        F.sum("bureau_current_months").alias("bureau_current_months"),
        F.sum("bureau_closed_months").alias("bureau_closed_months")
    )
)

print(
    f"Bureau application-level rows: "
    f"{bureau_agg.count():,}"
)

display(bureau_agg.limit(10))


# ============================================================
# Installments → Previous application level
# ============================================================

print("=" * 80)
print("AGGREGATING INSTALLMENT PAYMENTS")
print("=" * 80)

installments_agg = (
    bronze["installments_payments"]
    .groupBy("SK_ID_PREV")
    .agg(
        F.count("*").alias("installment_count"),
        
        F.sum("AMT_INSTALMENT").alias(
            "installment_total_due"
        ),
        
        F.sum("AMT_PAYMENT").alias(
            "installment_total_paid"
        ),
        
        F.avg("AMT_PAYMENT").alias(
            "installment_avg_payment"
        ),
        
        F.max("AMT_PAYMENT").alias(
            "installment_max_payment"
        ),
        
        F.sum(
            F.when(
                F.col("DAYS_ENTRY_PAYMENT") >
                F.col("DAYS_INSTALMENT"),
                1
            ).otherwise(0)
        ).alias("installment_late_count"),
        
        F.sum(
            F.when(
                F.col("DAYS_ENTRY_PAYMENT") <=
                F.col("DAYS_INSTALMENT"),
                1
            ).otherwise(0)
        ).alias("installment_on_time_count")
    )
)

print(
    f"Installment aggregate rows: "
    f"{installments_agg.count():,}"
)


# ============================================================
# POS Cash → Previous application level
# ============================================================

print("=" * 80)
print("AGGREGATING POS CASH HISTORY")
print("=" * 80)

pos_agg = (
    bronze["POS_CASH_balance"]
    .groupBy("SK_ID_PREV")
    .agg(
        F.count("*").alias("pos_month_count"),
        
        F.max("SK_DPD").alias("pos_max_dpd"),
        F.max("SK_DPD_DEF").alias("pos_max_dpd_def"),
        
        F.sum(
            F.when(F.col("SK_DPD") > 0, 1)
            .otherwise(0)
        ).alias("pos_dpd_months"),
        
        F.sum(
            F.when(F.col("SK_DPD_DEF") > 0, 1)
            .otherwise(0)
        ).alias("pos_dpd_def_months"),
        
        F.avg("CNT_INSTALMENT_FUTURE").alias(
            "pos_avg_installments_future"
        ),
        
        F.min("MONTHS_BALANCE").alias(
            "pos_oldest_month"
        ),
        
        F.max("MONTHS_BALANCE").alias(
            "pos_latest_month"
        )
    )
)

print(
    f"POS aggregate rows: "
    f"{pos_agg.count():,}"
)


# ============================================================
# Credit card → Previous application level
# ============================================================

print("=" * 80)
print("AGGREGATING CREDIT CARD HISTORY")
print("=" * 80)

credit_card_agg = (
    bronze["credit_card_balance"]
    .groupBy("SK_ID_PREV")
    .agg(
        F.count("*").alias("cc_month_count"),
        
        F.avg("AMT_BALANCE").alias(
            "cc_avg_balance"
        ),
        
        F.max("AMT_BALANCE").alias(
            "cc_max_balance"
        ),
        
        F.avg("AMT_CREDIT_LIMIT_ACTUAL").alias(
            "cc_avg_credit_limit"
        ),
        
        F.max("AMT_CREDIT_LIMIT_ACTUAL").alias(
            "cc_max_credit_limit"
        ),
        
        F.sum("AMT_DRAWINGS_CURRENT").alias(
            "cc_total_drawings"
        ),
        
        F.sum("AMT_PAYMENT_TOTAL_CURRENT").alias(
            "cc_total_payments"
        ),
        
        F.max("SK_DPD").alias(
            "cc_max_dpd"
        ),
        
        F.max("SK_DPD_DEF").alias(
            "cc_max_dpd_def"
        ),
        
        F.sum(
            F.when(F.col("SK_DPD") > 0, 1)
            .otherwise(0)
        ).alias("cc_dpd_months"),
        
        F.sum(
            F.when(F.col("SK_DPD_DEF") > 0, 1)
            .otherwise(0)
        ).alias("cc_dpd_def_months")
    )
)

print(
    f"Credit card aggregate rows: "
    f"{credit_card_agg.count():,}"
)


# ============================================================
# Previous applications + child history
# ============================================================

print("=" * 80)
print("ENRICHING PREVIOUS APPLICATIONS")
print("=" * 80)

previous_enriched = (
    bronze["previous_application"]
    .join(
        installments_agg,
        on="SK_ID_PREV",
        how="left"
    )
    .join(
        pos_agg,
        on="SK_ID_PREV",
        how="left"
    )
    .join(
        credit_card_agg,
        on="SK_ID_PREV",
        how="left"
    )
)


# ============================================================
# Previous applications → application/customer level
# ============================================================

previous_agg = (
    previous_enriched
    .groupBy("SK_ID_CURR")
    .agg(
        # Application history
        F.count("*").alias(
            "previous_application_count"
        ),
        
        F.sum(
            F.when(
                F.col("NAME_CONTRACT_STATUS") == "Approved",
                1
            ).otherwise(0)
        ).alias("previous_approved_count"),
        
        F.sum(
            F.when(
                F.col("NAME_CONTRACT_STATUS") == "Refused",
                1
            ).otherwise(0)
        ).alias("previous_refused_count"),
        
        # Historical credit amounts
        F.sum("AMT_APPLICATION").alias(
            "previous_total_requested"
        ),
        
        F.sum("AMT_CREDIT").alias(
            "previous_total_credit"
        ),
        
        F.avg("AMT_CREDIT").alias(
            "previous_avg_credit"
        ),
        
        F.max("AMT_CREDIT").alias(
            "previous_max_credit"
        ),
        
        # Application outcomes
        F.countDistinct(
            "NAME_CONTRACT_STATUS"
        ).alias(
            "previous_status_type_count"
        ),
        
        # Historical installment behaviour
        F.sum("installment_count").alias(
            "historical_installment_count"
        ),
        
        F.sum("installment_total_due").alias(
            "historical_total_due"
        ),
        
        F.sum("installment_total_paid").alias(
            "historical_total_paid"
        ),
        
        F.sum("installment_late_count").alias(
            "historical_late_payment_count"
        ),
        
        F.sum("installment_on_time_count").alias(
            "historical_on_time_payment_count"
        ),
        
        # POS delinquency
        F.sum("pos_dpd_months").alias(
            "historical_pos_dpd_months"
        ),
        
        F.sum("pos_dpd_def_months").alias(
            "historical_pos_dpd_def_months"
        ),
        
        F.max("pos_max_dpd").alias(
            "historical_max_pos_dpd"
        ),
        
        F.max("pos_max_dpd_def").alias(
            "historical_max_pos_dpd_def"
        ),
        
        # Credit-card behaviour
        F.avg("cc_avg_balance").alias(
            "historical_avg_cc_balance"
        ),
        
        F.max("cc_max_balance").alias(
            "historical_max_cc_balance"
        ),
        
        F.sum("cc_total_drawings").alias(
            "historical_cc_drawings"
        ),
        
        F.sum("cc_total_payments").alias(
            "historical_cc_payments"
        ),
        
        F.sum("cc_dpd_months").alias(
            "historical_cc_dpd_months"
        ),
        
        F.sum("cc_dpd_def_months").alias(
            "historical_cc_dpd_def_months"
        ),
        
        # Recency of previous applications
        F.max("DAYS_DECISION").alias(
            "most_recent_previous_application_days"
        ),
        
        F.min("DAYS_DECISION").alias(
            "oldest_previous_application_days"
        )
    )
)

print(
    f"Previous application aggregate rows: "
    f"{previous_agg.count():,}"
)

display(previous_agg.limit(10))


# ============================================================
# Combine application + historical aggregates
# ============================================================

print("=" * 80)
print("BUILDING APPLICATION-LEVEL FEATURE TABLE")
print("=" * 80)

application_train = bronze["application_train"]

application_features = (
    application_train
    .join(
        bureau_agg,
        on="SK_ID_CURR",
        how="left"
    )
    .join(
        previous_agg,
        on="SK_ID_CURR",
        how="left"
    )
)


# ============================================================
# Basic engineered ratios
# ============================================================

application_features = (
    application_features
    .withColumn(
        "bureau_debt_to_credit_ratio",
        F.when(
            F.col("bureau_total_credit") > 0,
            F.col("bureau_total_debt") /
            F.col("bureau_total_credit")
        )
    )
    .withColumn(
        "bureau_overdue_to_credit_ratio",
        F.when(
            F.col("bureau_total_credit") > 0,
            F.col("bureau_total_overdue") /
            F.col("bureau_total_credit")
        )
    )
    .withColumn(
        "previous_approval_rate",
        F.when(
            F.col("previous_application_count") > 0,
            F.col("previous_approved_count") /
            F.col("previous_application_count")
        )
    )
    .withColumn(
        "previous_refusal_rate",
        F.when(
            F.col("previous_application_count") > 0,
            F.col("previous_refused_count") /
            F.col("previous_application_count")
        )
    )
    .withColumn(
        "historical_payment_completion_ratio",
        F.when(
            F.col("historical_total_due") > 0,
            F.col("historical_total_paid") /
            F.col("historical_total_due")
        )
    )
    .withColumn(
        "historical_late_payment_rate",
        F.when(
            (
                F.col("historical_late_payment_count") +
                F.col("historical_on_time_payment_count")
            ) > 0,
            F.col("historical_late_payment_count") /
            (
                F.col("historical_late_payment_count") +
                F.col("historical_on_time_payment_count")
            )
        )
    )
)


# ============================================================
# Integrity checks
# ============================================================

print("=" * 80)
print("APPLICATION FEATURE TABLE INTEGRITY")
print("=" * 80)

total_rows = application_features.count()

distinct_customers = (
    application_features
    .select("SK_ID_CURR")
    .distinct()
    .count()
)

print(f"Rows: {total_rows:,}")
print(f"Distinct SK_ID_CURR: {distinct_customers:,}")
print(f"Columns: {len(application_features.columns)}")

if total_rows == distinct_customers:
    print("✓ Application grain preserved: one row per SK_ID_CURR")
else:
    print("⚠ WARNING: Application grain has been duplicated")


# ============================================================
# Historical feature coverage
# ============================================================

coverage_summary = application_features.select(
    F.count("*").alias("total_applications"),
    
    F.sum(
        F.when(
            F.col("bureau_account_count").isNotNull(),
            1
        ).otherwise(0)
    ).alias("applications_with_bureau_history"),
    
    F.sum(
        F.when(
            F.col("previous_application_count").isNotNull(),
            1
        ).otherwise(0)
    ).alias("applications_with_previous_history")
)

display(coverage_summary)


# ============================================================
# Save engineered application-level dataset
# ============================================================

SILVER_PATH = f"{BASE_PATH}/silver_application_features"

(
    application_features
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

print("=" * 80)
print("HOME CREDIT SILVER DATASET SAVED")
print("=" * 80)
print(SILVER_PATH)

AGGREGATING BUREAU BALANCE
Bureau balance aggregate rows: 817,395


SK_ID_BUREAU,bureau_balance_months,bureau_delinquent_months,bureau_current_months,bureau_closed_months,bureau_unknown_months
6146819,59,0,4,0,55
6146824,29,0,0,0,29
6146839,52,0,15,37,0
6146848,7,0,7,0,0
6146892,4,0,4,0,0
6146896,15,5,10,0,0
6147403,58,0,9,48,1
6147539,49,0,4,39,6
6147752,67,0,27,40,0
6148107,29,0,8,21,0


AGGREGATING BUREAU HISTORY TO APPLICATION LEVEL
Bureau application-level rows: 305,811


SK_ID_CURR,bureau_account_count,bureau_active_count,bureau_closed_count,bureau_total_credit,bureau_total_debt,bureau_total_overdue,bureau_max_credit,bureau_max_debt,bureau_max_overdue,bureau_max_days_overdue,bureau_overdue_account_count,bureau_oldest_credit_days,bureau_most_recent_credit_days,bureau_credit_type_count,bureau_currency_count,bureau_total_prolongations,bureau_balance_months,bureau_delinquent_months,bureau_current_months,bureau_closed_months
117612,7,3,4,3179133.0,0.0,0.0,2056500.0,0.0,0.0,0,0,-1800,-419,2,1,0,null,null,null,null
121827,18,6,12,1.095975E7,368959.5,0.0,2362500.0,368959.5,0.0,0,0,-2838,-402,3,1,0,639,0,266,282
257103,7,3,3,1099858.5,140526.0,0.0,247500.0,140526.0,0.0,0,0,-2594,-62,2,1,1,null,null,null,null
197424,9,3,6,2237702.805,831001.5,0.0,540000.0,386473.5,0.0,0,0,-2549,-430,1,1,0,null,null,null,null
328589,3,1,2,198655.74,90700.92,0.0,100271.25,90700.92,0.0,0,0,-425,-295,2,1,0,null,null,null,null
401899,7,4,3,1030276.08,690374.565,0.0,675000.0,507820.5,0.0,0,0,-1597,-21,2,1,0,175,1,55,25
128936,7,4,3,1748638.8,1105528.5,0.0,765000.0,717970.5,0.0,0,0,-646,-26,2,1,0,54,0,7,17
262719,3,0,3,280098.0,0.0,0.0,139050.0,0.0,0.0,0,0,-1300,-308,1,1,0,59,1,23,35
221762,3,0,3,447723.0,0.0,0.0,203107.5,0.0,0.0,0,0,-2139,-549,1,1,0,136,1,18,91
448942,13,7,6,3938483.565,1619707.5,0.0,686700.0,632146.5,0.0,0,0,-1395,-99,2,1,0,null,null,null,null


AGGREGATING INSTALLMENT PAYMENTS
Installment aggregate rows: 997,752
AGGREGATING POS CASH HISTORY
POS aggregate rows: 936,325
AGGREGATING CREDIT CARD HISTORY
Credit card aggregate rows: 104,307
ENRICHING PREVIOUS APPLICATIONS
Previous application aggregate rows: 338,857


SK_ID_CURR,previous_application_count,previous_approved_count,previous_refused_count,previous_total_requested,previous_total_credit,previous_avg_credit,previous_max_credit,previous_status_type_count,historical_installment_count,historical_total_due,historical_total_paid,historical_late_payment_count,historical_on_time_payment_count,historical_pos_dpd_months,historical_pos_dpd_def_months,historical_max_pos_dpd,historical_max_pos_dpd_def,historical_avg_cc_balance,historical_max_cc_balance,historical_cc_drawings,historical_cc_payments,historical_cc_dpd_months,historical_cc_dpd_def_months,most_recent_previous_application_days,oldest_previous_application_days
357641,6,5,1,698778.0,882994.5,147165.75,241920.0,2,42,668791.35,668791.35,0,42,0,0,0,0,null,null,null,null,null,null,-775,-2799
391198,4,3,1,291181.5,283198.5,70799.625,131980.5,2,13,268314.52499999997,268314.52499999997,0,13,0,0,0,0,null,null,null,null,null,null,-164,-1584
120860,6,4,2,1202445.0,1331932.5,221988.75,787500.0,2,106,3137233.2300000004,3134983.23,8,98,1,1,1,1,164235.32052631577,472388.625,1263843.495,1438323.93,0,0,-283,-1509
412390,2,2,0,159480.0,172548.0,86274.0,101943.0,1,9,124368.70499999999,82910.43000000001,4,5,0,0,0,0,null,null,null,null,null,null,-18,-1579
295864,10,4,5,815071.5,1480077.0,148007.7,450000.0,3,63,460519.01999999996,428426.55,8,55,0,0,0,0,null,null,null,null,null,null,-237,-2480
410586,12,8,3,1093819.5,1444144.5,120345.375,180427.5,3,68,798909.84,715638.645,13,55,0,0,0,0,null,null,null,null,null,null,-158,-2440
205463,5,4,1,170140.5,387130.5,77426.1,225000.0,2,49,320826.24,207908.73,34,15,12,11,36,20,null,null,null,null,null,null,-798,-1233
332676,10,7,0,576372.15,610011.0,61001.1,225000.0,2,60,318127.5,344074.5,0,60,0,0,0,0,0.0,0.0,0.0,0.0,0,0,-52,-1593
257794,5,3,0,143599.5,140485.5,28097.1,67495.5,2,67,247010.35500000004,237442.72500000003,5,62,1,1,21,21,47103.260294117645,78076.62,121706.14499999999,102624.56999999999,0,0,-45,-1379
397057,3,1,1,1030486.5,994936.5,331645.5,675000.0,3,9,359110.575,359110.575,0,9,0,0,0,0,null,null,null,null,null,null,-421,-556


BUILDING APPLICATION-LEVEL FEATURE TABLE
APPLICATION FEATURE TABLE INTEGRITY
Rows: 307,511
Distinct SK_ID_CURR: 307,511
Columns: 173
✓ Application grain preserved: one row per SK_ID_CURR


total_applications,applications_with_bureau_history,applications_with_previous_history
307511,263491,291057


HOME CREDIT SILVER DATASET SAVED
/Volumes/workspace/default/home_credit_raw/silver_application_features


## Silver Feature Quality and Leakage Audit

The application-level Silver dataset is validated before modelling.

The audit examines:

- feature completeness
- structural missingness created by absent historical records
- numerical feature ranges
- engineered ratio validity
- target leakage
- preservation of the application-level grain

Historical features are constructed from the Home Credit historical tables and are kept separate from the TARGET variable.

Missing historical aggregates are interpreted as absence of available historical records rather than automatically being treated as data-quality errors. Final imputation and modelling decisions are deferred to the model-preparation notebook.

In [0]:
from pyspark.sql import functions as F


# ============================================================
# Reload Silver application-level feature table
# ============================================================

SILVER_PATH = "/Volumes/workspace/default/home_credit_raw/silver_application_features"

hc_silver = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
)


# ============================================================
# Basic integrity
# ============================================================

print("=" * 80)
print("SILVER DATASET INTEGRITY")
print("=" * 80)

row_count = hc_silver.count()

distinct_customers = (
    hc_silver
    .select("SK_ID_CURR")
    .distinct()
    .count()
)

print(f"Rows: {row_count:,}")
print(f"Distinct SK_ID_CURR: {distinct_customers:,}")
print(f"Columns: {len(hc_silver.columns)}")

if row_count == distinct_customers:
    print("✓ One row per application/customer")
else:
    print("⚠ WARNING: Duplicate application grain detected")


# ============================================================
# Target leakage audit
# ============================================================

print("\n" + "=" * 80)
print("TARGET LEAKAGE AUDIT")
print("=" * 80)

target_columns = [
    c for c in hc_silver.columns
    if "TARGET" in c.upper()
]

print("Columns containing TARGET:")
print(target_columns)

if target_columns == ["TARGET"]:
    print("✓ TARGET appears only as the modelling outcome")
else:
    print("⚠ Review TARGET-related columns")


# ============================================================
# Missingness profile
# ============================================================

print("\n" + "=" * 80)
print("FEATURE MISSINGNESS PROFILE")
print("=" * 80)

total_rows = hc_silver.count()

missingness_rows = []

for column_name in hc_silver.columns:

    missing_count = (
        hc_silver
        .filter(F.col(column_name).isNull())
        .count()
    )

    missing_pct = (
        missing_count / total_rows * 100
        if total_rows > 0
        else 0
    )

    missingness_rows.append(
        (
            column_name,
            missing_count,
            round(missing_pct, 3)
        )
    )

missingness_df = spark.createDataFrame(
    missingness_rows,
    [
        "column_name",
        "missing_count",
        "missing_pct"
    ]
)

display(
    missingness_df
    .orderBy(F.desc("missing_pct"))
)


# ============================================================
# Historical feature coverage
# ============================================================

print("\n" + "=" * 80)
print("HISTORICAL FEATURE COVERAGE")
print("=" * 80)

historical_columns = [
    "bureau_account_count",
    "bureau_total_credit",
    "bureau_total_debt",
    "bureau_total_overdue",
    "previous_application_count",
    "previous_approved_count",
    "previous_refused_count",
    "historical_installment_count",
    "historical_total_due",
    "historical_total_paid",
    "historical_late_payment_count",
    "historical_pos_dpd_months",
    "historical_max_pos_dpd",
    "historical_avg_cc_balance",
    "historical_max_cc_balance",
    "historical_cc_dpd_months"
]

coverage_rows = []

for column_name in historical_columns:

    if column_name in hc_silver.columns:

        available = (
            hc_silver
            .filter(F.col(column_name).isNotNull())
            .count()
        )

        coverage_pct = available / total_rows * 100

        coverage_rows.append(
            (
                column_name,
                available,
                round(coverage_pct, 3)
            )
        )

coverage_df = spark.createDataFrame(
    coverage_rows,
    [
        "feature",
        "non_null_rows",
        "coverage_pct"
    ]
)

display(
    coverage_df
    .orderBy(F.desc("coverage_pct"))
)


# ============================================================
# Engineered ratio diagnostics
# ============================================================

print("\n" + "=" * 80)
print("ENGINEERED RATIO DIAGNOSTICS")
print("=" * 80)

ratio_columns = [
    "bureau_debt_to_credit_ratio",
    "bureau_overdue_to_credit_ratio",
    "previous_approval_rate",
    "previous_refusal_rate",
    "historical_payment_completion_ratio",
    "historical_late_payment_rate"
]

ratio_summary_rows = []

for column_name in ratio_columns:

    if column_name in hc_silver.columns:

        stats = (
            hc_silver
            .select(
                F.min(column_name).alias("min_value"),
                F.max(column_name).alias("max_value"),
                F.avg(column_name).alias("mean_value"),
                F.expr(f"percentile_approx({column_name}, 0.5)").alias("median_value"),
                F.sum(
                    F.when(F.col(column_name).isNull(), 1)
                    .otherwise(0)
                ).alias("null_count")
            )
            .collect()[0]
        )

        ratio_summary_rows.append(
            (
                column_name,
                stats["min_value"],
                stats["max_value"],
                stats["mean_value"],
                stats["median_value"],
                stats["null_count"]
            )
        )

ratio_summary_df = spark.createDataFrame(
    ratio_summary_rows,
    [
        "feature",
        "min_value",
        "max_value",
        "mean_value",
        "median_value",
        "null_count"
    ]
)

display(ratio_summary_df)


# ============================================================
# Check theoretically bounded ratios
# ============================================================

print("\n" + "=" * 80)
print("BOUNDED RATIO CHECK")
print("=" * 80)

bounded_checks = []

# Approval and refusal rates should normally lie between 0 and 1
for column_name in [
    "previous_approval_rate",
    "previous_refusal_rate",
    "historical_late_payment_rate"
]:

    invalid_count = (
        hc_silver
        .filter(
            (F.col(column_name) < 0) |
            (F.col(column_name) > 1)
        )
        .count()
    )

    bounded_checks.append(
        (
            column_name,
            invalid_count
        )
    )

bounded_df = spark.createDataFrame(
    bounded_checks,
    [
        "feature",
        "invalid_out_of_range_count"
    ]
)

display(bounded_df)


# ============================================================
# Extreme-value diagnostics for key historical variables
# ============================================================

print("\n" + "=" * 80)
print("KEY HISTORICAL FEATURE SUMMARY")
print("=" * 80)

summary_columns = [
    "bureau_account_count",
    "bureau_total_credit",
    "bureau_total_debt",
    "bureau_total_overdue",
    "bureau_max_days_overdue",
    "previous_application_count",
    "previous_total_credit",
    "historical_installment_count",
    "historical_total_due",
    "historical_total_paid",
    "historical_late_payment_count",
    "historical_max_pos_dpd",
    "historical_max_pos_dpd_def",
    "historical_avg_cc_balance",
    "historical_max_cc_balance"
]

summary_columns = [
    c for c in summary_columns
    if c in hc_silver.columns
]

display(
    hc_silver
    .select(summary_columns)
    .summary(
        "count",
        "mean",
        "stddev",
        "min",
        "50%",
        "max"
    )
)


# ============================================================
# Target distribution re-check
# ============================================================

print("\n" + "=" * 80)
print("TARGET DISTRIBUTION")
print("=" * 80)

display(
    hc_silver
    .groupBy("TARGET")
    .count()
    .withColumn(
        "percentage",
        F.round(
            F.col("count") / row_count * 100,
            3
        )
    )
    .orderBy("TARGET")
)


# ============================================================
# Final Silver validation
# ============================================================

print("\n" + "=" * 80)
print("SILVER FEATURE QUALITY AUDIT COMPLETE")
print("=" * 80)

print(f"Application rows: {row_count:,}")
print(f"Unique applications: {distinct_customers:,}")
print(f"Total columns: {len(hc_silver.columns)}")
print("Target leakage audit completed.")
print("Missingness profile completed.")
print("Historical feature coverage completed.")
print("Ratio diagnostics completed.")
print("Silver dataset is ready for model preparation after review.")

SILVER DATASET INTEGRITY
Rows: 307,511
Distinct SK_ID_CURR: 307,511
Columns: 173
✓ One row per application/customer

TARGET LEAKAGE AUDIT
Columns containing TARGET:
['TARGET']
✓ TARGET appears only as the modelling outcome

FEATURE MISSINGNESS PROFILE


column_name,missing_count,missing_pct
historical_avg_cc_balance,229577,74.657
historical_max_cc_balance,229577,74.657
historical_cc_drawings,229577,74.657
historical_cc_payments,229577,74.657
historical_cc_dpd_months,229577,74.657
historical_cc_dpd_def_months,229577,74.657
bureau_balance_months,215280,70.007
bureau_delinquent_months,215280,70.007
bureau_current_months,215280,70.007
bureau_closed_months,215280,70.007



HISTORICAL FEATURE COVERAGE


feature,non_null_rows,coverage_pct
previous_application_count,291057,94.649
previous_approved_count,291057,94.649
previous_refused_count,291057,94.649
historical_installment_count,289406,94.112
historical_total_due,289406,94.112
historical_late_payment_count,289406,94.112
historical_total_paid,289398,94.11
historical_pos_dpd_months,286967,93.319
historical_max_pos_dpd,286967,93.319
bureau_account_count,263491,85.685



ENGINEERED RATIO DIAGNOSTICS


...


KEY HISTORICAL FEATURE SUMMARY


summary,bureau_account_count,bureau_total_credit,bureau_total_debt,bureau_total_overdue,bureau_max_days_overdue,previous_application_count,previous_total_credit,historical_installment_count,historical_total_due,historical_total_paid,historical_late_payment_count,historical_max_pos_dpd,historical_max_pos_dpd_def,historical_avg_cc_balance,historical_max_cc_balance
count,263491,263490,256131,263491,263491,291057,291057,289406,289406,289398,289406,286967,286967,77934,77934
mean,5.56119563856071,1955814.0087521141,659059.5311415012,223.03434088830357,4.772758841857976,4.857127641664691,953716.1846199703,36.53075955577977,660247.5274753472,666989.0246191892,3.2353406632896347,9.270114682175999,1.006652332846634,75634.6283839219,147943.77239048414
stddev,4.377896725265539,4101733.795513972,1653606.385583858,16497.193326265882,89.14127391336557,4.147042050313135,1486623.0608670353,35.91811240353781,864203.4297779471,901313.1411376994,6.162680077827859,96.96251003478446,4.400579908813906,113612.33821685438,180048.7057027399
min,1,0.0,-6981558.210000001,0.0,0,1,0.0,1,0.0,47.7,0,0,0,-2930.232558139535,0.0
50%,4,961650.0,184487.40000000002,0.0,0,4,427765.5,24,322285.41,313041.33,1,0,0,28866.6,96645.015
max,116,1.017957917385E9,3.3449833120500004E8,3756681.0,2792,73,4.1461128E7,350,2.3274726930000003E7,2.5537053780000005E7,154,2743,1117,928686.3235714287,1354829.265



TARGET DISTRIBUTION


TARGET,count,percentage
0,282686,91.927
1,24825,8.073



SILVER FEATURE QUALITY AUDIT COMPLETE
Application rows: 307,511
Unique applications: 307,511
Total columns: 173
Target leakage audit completed.
Missingness profile completed.
Historical feature coverage completed.
Ratio diagnostics completed.
Silver dataset is ready for model preparation after review.


## Historical Feature Anomaly Audit

The aggregated historical features are inspected for economically unusual or mathematically inconsistent values before finalizing the Silver dataset.

The audit focuses on:

- negative credit and debt amounts
- unusually high debt-to-credit ratios
- unusually high overdue-to-credit ratios
- payment amounts exceeding cumulative installment amounts
- extreme delinquency values
- the distinction between source-data anomalies and structural missingness

No values are modified during this diagnostic stage. Treatment decisions are deferred until the anomaly patterns have been quantified.

In [0]:
from pyspark.sql import functions as F


# ============================================================
# Anomaly audit
# ============================================================

print("=" * 80)
print("HISTORICAL FEATURE ANOMALY AUDIT")
print("=" * 80)


# ============================================================
# Helper function
# ============================================================

def count_condition(df, condition):
    return df.filter(condition).count()


# ============================================================
# Bureau monetary anomalies
# ============================================================

bureau_checks = [
    (
        "bureau_total_credit < 0",
        F.col("bureau_total_credit") < 0
    ),
    (
        "bureau_total_debt < 0",
        F.col("bureau_total_debt") < 0
    ),
    (
        "bureau_total_overdue < 0",
        F.col("bureau_total_overdue") < 0
    ),
    (
        "bureau_max_credit < 0",
        F.col("bureau_max_credit") < 0
    ),
    (
        "bureau_max_debt < 0",
        F.col("bureau_max_debt") < 0
    ),
    (
        "bureau_max_overdue < 0",
        F.col("bureau_max_overdue") < 0
    ),
]

bureau_anomaly_rows = []

for description, condition in bureau_checks:

    count = count_condition(
        hc_silver,
        condition
    )

    bureau_anomaly_rows.append(
        (
            description,
            count,
            round(count / row_count * 100, 4)
        )
    )

bureau_anomaly_df = spark.createDataFrame(
    bureau_anomaly_rows,
    [
        "condition",
        "row_count",
        "percentage_of_applications"
    ]
)

display(bureau_anomaly_df)


# ============================================================
# Bureau ratio anomaly counts
# ============================================================

ratio_checks = [
    (
        "bureau_debt_to_credit_ratio < 0",
        F.col("bureau_debt_to_credit_ratio") < 0
    ),
    (
        "bureau_debt_to_credit_ratio > 1",
        F.col("bureau_debt_to_credit_ratio") > 1
    ),
    (
        "bureau_overdue_to_credit_ratio < 0",
        F.col("bureau_overdue_to_credit_ratio") < 0
    ),
    (
        "bureau_overdue_to_credit_ratio > 1",
        F.col("bureau_overdue_to_credit_ratio") > 1
    ),
    (
        "historical_payment_completion_ratio < 0",
        F.col("historical_payment_completion_ratio") < 0
    ),
    (
        "historical_payment_completion_ratio > 1",
        F.col("historical_payment_completion_ratio") > 1
    ),
]

ratio_anomaly_rows = []

for description, condition in ratio_checks:

    count = count_condition(
        hc_silver,
        condition
    )

    ratio_anomaly_rows.append(
        (
            description,
            count,
            round(count / row_count * 100, 4)
        )
    )

ratio_anomaly_df = spark.createDataFrame(
    ratio_anomaly_rows,
    [
        "condition",
        "row_count",
        "percentage_of_applications"
    ]
)

display(ratio_anomaly_df)


# ============================================================
# Payment consistency audit
# ============================================================

print("=" * 80)
print("PAYMENT CONSISTENCY AUDIT")
print("=" * 80)

payment_anomaly = (
    hc_silver
    .filter(
        (F.col("historical_total_due").isNotNull()) &
        (F.col("historical_total_paid").isNotNull()) &
        (
            F.col("historical_total_paid") >
            F.col("historical_total_due")
        )
    )
    .count()
)

payment_anomaly_pct = (
    payment_anomaly / row_count * 100
)

print(
    f"Applications where historical paid > historical due: "
    f"{payment_anomaly:,}"
)

print(
    f"Percentage of all applications: "
    f"{payment_anomaly_pct:.4f}%"
)


# ============================================================
# Credit-card balance anomalies
# ============================================================

print("=" * 80)
print("CREDIT CARD VALUE AUDIT")
print("=" * 80)

cc_checks = [
    (
        "historical_avg_cc_balance < 0",
        F.col("historical_avg_cc_balance") < 0
    ),
    (
        "historical_max_cc_balance < 0",
        F.col("historical_max_cc_balance") < 0
    ),
    (
        "historical_max_pos_dpd < 0",
        F.col("historical_max_pos_dpd") < 0
    ),
    (
        "historical_max_pos_dpd_def < 0",
        F.col("historical_max_pos_dpd_def") < 0
    ),
]

cc_anomaly_rows = []

for description, condition in cc_checks:

    count = count_condition(
        hc_silver,
        condition
    )

    cc_anomaly_rows.append(
        (
            description,
            count,
            round(count / row_count * 100, 4)
        )
    )

cc_anomaly_df = spark.createDataFrame(
    cc_anomaly_rows,
    [
        "condition",
        "row_count",
        "percentage_of_applications"
    ]
)

display(cc_anomaly_df)


# ============================================================
# Extreme delinquency audit
# ============================================================

print("=" * 80)
print("EXTREME DELINQUENCY AUDIT")
print("=" * 80)

dpd_summary = (
    hc_silver
    .select(
        F.expr(
            "percentile_approx(historical_max_pos_dpd, "
            "array(0.50, 0.90, 0.95, 0.99, 0.999), 10000)"
        ).alias("pos_dpd_percentiles"),
        
        F.expr(
            "percentile_approx(historical_max_pos_dpd_def, "
            "array(0.50, 0.90, 0.95, 0.99, 0.999), 10000)"
        ).alias("pos_dpd_def_percentiles"),
        
        F.expr(
            "percentile_approx(bureau_max_days_overdue, "
            "array(0.50, 0.90, 0.95, 0.99, 0.999), 10000)"
        ).alias("bureau_overdue_percentiles")
    )
)

display(dpd_summary)


# ============================================================
# Extreme monetary feature percentiles
# ============================================================

print("=" * 80)
print("EXTREME MONETARY FEATURE PERCENTILES")
print("=" * 80)

monetary_percentiles = (
    hc_silver
    .select(
        F.expr(
            "percentile_approx(bureau_total_credit, "
            "array(0.50, 0.90, 0.95, 0.99, 0.999), 10000)"
        ).alias("bureau_total_credit"),
        
        F.expr(
            "percentile_approx(bureau_total_debt, "
            "array(0.50, 0.90, 0.95, 0.99, 0.999), 10000)"
        ).alias("bureau_total_debt"),
        
        F.expr(
            "percentile_approx(previous_total_credit, "
            "array(0.50, 0.90, 0.95, 0.99, 0.999), 10000)"
        ).alias("previous_total_credit"),
        
        F.expr(
            "percentile_approx(historical_total_due, "
            "array(0.50, 0.90, 0.95, 0.99, 0.999), 10000)"
        ).alias("historical_total_due")
    )
)

display(monetary_percentiles)


# ============================================================
# Top extreme observations
# ============================================================

print("=" * 80)
print("MOST EXTREME BUREAU RATIO OBSERVATIONS")
print("=" * 80)

display(
    hc_silver
    .select(
        "SK_ID_CURR",
        "bureau_total_credit",
        "bureau_total_debt",
        "bureau_total_overdue",
        "bureau_debt_to_credit_ratio",
        "bureau_overdue_to_credit_ratio"
    )
    .orderBy(
        F.desc(
            F.abs(
                F.col("bureau_debt_to_credit_ratio")
            )
        )
    )
    .limit(20)
)


# ============================================================
# Final diagnostic summary
# ============================================================

print("=" * 80)
print("ANOMALY AUDIT COMPLETE")
print("=" * 80)

print("No source values have been modified.")
print("The next step is to determine robust feature treatment")
print("based on the observed anomaly frequency and magnitude.")

HISTORICAL FEATURE ANOMALY AUDIT


condition,row_count,percentage_of_applications
bureau_total_credit < 0,0,0.0
bureau_total_debt < 0,1296,0.4214
bureau_total_overdue < 0,0,0.0
bureau_max_credit < 0,0,0.0
bureau_max_debt < 0,85,0.0276
bureau_max_overdue < 0,0,0.0


condition,row_count,percentage_of_applications
bureau_debt_to_credit_ratio < 0,1265,0.4114
bureau_debt_to_credit_ratio > 1,1585,0.5154
bureau_overdue_to_credit_ratio < 0,0,0.0
bureau_overdue_to_credit_ratio > 1,7,0.0023
historical_payment_completion_ratio < 0,0,0.0
historical_payment_completion_ratio > 1,33380,10.8549


PAYMENT CONSISTENCY AUDIT
Applications where historical paid > historical due: 33,382
Percentage of all applications: 10.8555%
CREDIT CARD VALUE AUDIT


condition,row_count,percentage_of_applications
historical_avg_cc_balance < 0,28,0.0091
historical_max_cc_balance < 0,0,0.0
historical_max_pos_dpd < 0,0,0.0
historical_max_pos_dpd_def < 0,0,0.0


...

EXTREME MONETARY FEATURE PERCENTILES


bureau_total_credit,bureau_total_debt,previous_total_credit,historical_total_due
"List(961650.0, 4671738.0, 6984108.720000001, 1.411145595E7, 3.18888E7)","List(184487.40000000002, 1639435.5, 2701998.0, 7108228.395, 1.5972336E7)","List(427765.5, 2395287.0, 6951136.5, 1.4302188E7)","List(322285.41, 1730077.02, 2457448.965, 4081946.8950000005, 6377009.625000001)"


MOST EXTREME BUREAU RATIO OBSERVATIONS


SK_ID_CURR,bureau_total_credit,bureau_total_debt,bureau_total_overdue,bureau_debt_to_credit_ratio,bureau_overdue_to_credit_ratio
244750,4504.5,-789590.34,0.0,-175.28923076923076,0.0
205348,45000.0,350509.5,0.0,7.7891,0.0
449785,67634.45999999999,473011.425,0.0,6.993645325178911,0.0
136801,80959.095,-483478.335,0.0,-5.971884134821418,0.0
209349,45000.0,264780.0,0.0,5.884,0.0
292622,2055982.5,1.2012291E7,0.0,5.842603718660056,0.0
292568,135000.0,661167.0,0.0,4.8975333333333335,0.0
325198,22500.0,105880.5,0.0,4.7058,0.0
259372,19080.0,89558.235,0.0,4.693827830188679,0.0
103635,22500.0,95818.5,0.0,4.2586,0.0


ANOMALY AUDIT COMPLETE
No source values have been modified.
The next step is to determine robust feature treatment
based on the observed anomaly frequency and magnitude.


## Robust Treatment of Engineered Historical Features

The anomaly audit identified a small number of negative bureau debt values and extreme derived debt ratios, together with a substantial proportion of payment-completion ratios above one.

Raw Bronze values are retained unchanged.

For modelling-oriented derived features:

- negative bureau debt values are treated as invalid for ratio construction
- debt-to-credit and overdue-to-credit ratios are bounded to [0, 1]
- payment completion is capped at 1 because values above 1 represent payment exceeding scheduled installment amounts rather than additional completion
- an overpayment indicator is retained to preserve information contained in values above 1
- highly skewed delinquency and monetary variables are retained and will be transformed during model preparation

This approach separates source-data preservation from robust feature construction.

In [0]:
from pyspark.sql import functions as F


# ============================================================
# Reload current Silver dataset
# ============================================================

SILVER_PATH = "/Volumes/workspace/default/home_credit_raw/silver_application_features"

hc_silver = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
)


# ============================================================
# Robust bureau debt features
# ============================================================

hc_silver_clean = (
    hc_silver

    # --------------------------------------------------------
    # Clean bureau debt
    # Negative aggregate debt is not used as a valid balance
    # for derived risk ratios.
    # --------------------------------------------------------
    .withColumn(
        "bureau_total_debt_clean",
        F.when(
            F.col("bureau_total_debt") >= 0,
            F.col("bureau_total_debt")
        )
    )

    .withColumn(
        "bureau_max_debt_clean",
        F.when(
            F.col("bureau_max_debt") >= 0,
            F.col("bureau_max_debt")
        )
    )

    # --------------------------------------------------------
    # Robust debt-to-credit ratio
    # --------------------------------------------------------
    .withColumn(
        "bureau_debt_to_credit_ratio_clean",
        F.when(
            (F.col("bureau_total_credit") > 0) &
            (F.col("bureau_total_debt_clean") >= 0),
            F.least(
                F.lit(1.0),
                F.col("bureau_total_debt_clean") /
                F.col("bureau_total_credit")
            )
        )
    )

    # --------------------------------------------------------
    # Robust overdue-to-credit ratio
    # --------------------------------------------------------
    .withColumn(
        "bureau_overdue_to_credit_ratio_clean",
        F.when(
            (F.col("bureau_total_credit") > 0) &
            (F.col("bureau_total_overdue") >= 0),
            F.least(
                F.lit(1.0),
                F.col("bureau_total_overdue") /
                F.col("bureau_total_credit")
            )
        )
    )

    # ========================================================
    # Payment completion / overpayment
    # ========================================================

    .withColumn(
        "historical_payment_completion_ratio_clean",
        F.when(
            F.col("historical_total_due") > 0,
            F.least(
                F.lit(1.0),
                F.col("historical_total_paid") /
                F.col("historical_total_due")
            )
        )
    )

    .withColumn(
        "historical_overpayment_ratio",
        F.when(
            F.col("historical_total_due") > 0,
            F.greatest(
                F.lit(0.0),
                (
                    F.col("historical_total_paid") /
                    F.col("historical_total_due")
                ) - F.lit(1.0)
            )
        )
    )

    .withColumn(
        "historical_overpayment_flag",
        F.when(
            F.col("historical_overpayment_ratio") > 0,
            F.lit(1)
        ).otherwise(F.lit(0))
    )
)


# ============================================================
# Keep original features, but create clear cleaned versions
# ============================================================

# We intentionally keep the original aggregate columns.
# The cleaned derived columns will be preferred during
# model preparation.


# ============================================================
# Validate cleaned ratios
# ============================================================

print("=" * 80)
print("CLEANED FEATURE VALIDATION")
print("=" * 80)


clean_ratio_checks = [
    (
        "bureau_debt_to_credit_ratio_clean",
        F.col("bureau_debt_to_credit_ratio_clean")
    ),
    (
        "bureau_overdue_to_credit_ratio_clean",
        F.col("bureau_overdue_to_credit_ratio_clean")
    ),
    (
        "historical_payment_completion_ratio_clean",
        F.col("historical_payment_completion_ratio_clean")
    )
]

clean_ratio_results = []

for feature_name, column_expression in clean_ratio_checks:

    invalid_count = (
        hc_silver_clean
        .filter(
            (column_expression < 0) |
            (column_expression > 1)
        )
        .count()
    )

    clean_ratio_results.append(
        (
            feature_name,
            invalid_count
        )
    )

clean_ratio_df = spark.createDataFrame(
    clean_ratio_results,
    [
        "feature",
        "invalid_out_of_range_count"
    ]
)

display(clean_ratio_df)


# ============================================================
# Validate negative cleaned debt
# ============================================================

print("=" * 80)
print("CLEANED BUREAU DEBT VALIDATION")
print("=" * 80)

negative_clean_debt = (
    hc_silver_clean
    .filter(
        F.col("bureau_total_debt_clean") < 0
    )
    .count()
)

negative_clean_max_debt = (
    hc_silver_clean
    .filter(
        F.col("bureau_max_debt_clean") < 0
    )
    .count()
)

print(
    f"Negative bureau_total_debt_clean: "
    f"{negative_clean_debt:,}"
)

print(
    f"Negative bureau_max_debt_clean: "
    f"{negative_clean_max_debt:,}"
)


# ============================================================
# Overpayment profile
# ============================================================

print("=" * 80)
print("OVERPAYMENT PROFILE")
print("=" * 80)

overpayment_summary = (
    hc_silver_clean
    .select(
        F.sum(
            F.when(
                F.col("historical_overpayment_flag") == 1,
                1
            ).otherwise(0)
        ).alias("applications_with_overpayment"),

        F.avg(
            F.when(
                F.col("historical_overpayment_flag") == 1,
                F.col("historical_overpayment_ratio")
            )
        ).alias("average_overpayment_ratio_among_overpaid"),

        F.max(
            "historical_overpayment_ratio"
        ).alias("maximum_overpayment_ratio")
    )
)

display(overpayment_summary)


# ============================================================
# Feature inventory after robust treatment
# ============================================================

print("=" * 80)
print("SILVER FEATURE INVENTORY")
print("=" * 80)

print(f"Rows: {hc_silver_clean.count():,}")
print(f"Columns: {len(hc_silver_clean.columns)}")

distinct_ids = (
    hc_silver_clean
    .select("SK_ID_CURR")
    .distinct()
    .count()
)

print(f"Distinct SK_ID_CURR: {distinct_ids:,}")

if distinct_ids == hc_silver_clean.count():
    print("✓ One row per application preserved")
else:
    print("⚠ WARNING: Application grain changed")


# ============================================================
# Save cleaned Silver dataset
# ============================================================

CLEAN_SILVER_PATH = (
    "/Volumes/workspace/default/home_credit_raw/"
    "silver_application_features_clean"
)

(
    hc_silver_clean
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(CLEAN_SILVER_PATH)
)


# ============================================================
# Reload verification
# ============================================================

hc_silver_clean_verified = (
    spark.read
    .format("delta")
    .load(CLEAN_SILVER_PATH)
)

print("=" * 80)
print("CLEAN SILVER DATASET SAVED")
print("=" * 80)

print(CLEAN_SILVER_PATH)
print(f"Verified rows: {hc_silver_clean_verified.count():,}")
print(f"Verified columns: {len(hc_silver_clean_verified.columns)}")

CLEANED FEATURE VALIDATION


feature,invalid_out_of_range_count
bureau_debt_to_credit_ratio_clean,0
bureau_overdue_to_credit_ratio_clean,0
historical_payment_completion_ratio_clean,0


CLEANED BUREAU DEBT VALIDATION
Negative bureau_total_debt_clean: 0
Negative bureau_max_debt_clean: 0
OVERPAYMENT PROFILE


applications_with_overpayment,average_overpayment_ratio_among_overpaid,maximum_overpayment_ratio
33380,0.25048096433579325,4.485141019524729


SILVER FEATURE INVENTORY
Rows: 307,511
Columns: 180
Distinct SK_ID_CURR: 307,511
✓ One row per application preserved
CLEAN SILVER DATASET SAVED
/Volumes/workspace/default/home_credit_raw/silver_application_features_clean
Verified rows: 307,511
Verified columns: 180


In [0]:
from pyspark.sql import functions as F

# ============================================================
# FINAL HOME CREDIT SILVER QUALITY GATE
# ============================================================

print("=" * 80)
print("FINAL HOME CREDIT SILVER QUALITY GATE")
print("=" * 80)


# ============================================================
# Load verified cleaned Silver dataset
# ============================================================

CLEAN_SILVER_PATH = (
    "/Volumes/workspace/default/home_credit_raw/"
    "silver_application_features_clean"
)

hc_silver_final = (
    spark.read
    .format("delta")
    .load(CLEAN_SILVER_PATH)
)


# ============================================================
# Core dataset checks
# ============================================================

row_count = hc_silver_final.count()

column_count = len(
    hc_silver_final.columns
)

distinct_customer_ids = (
    hc_silver_final
    .select("SK_ID_CURR")
    .distinct()
    .count()
)

duplicate_customer_ids = (
    hc_silver_final
    .groupBy("SK_ID_CURR")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


# ============================================================
# Target and required-column checks
# ============================================================

target_present = (
    "TARGET" in hc_silver_final.columns
)

required_columns = [
    "SK_ID_CURR",
    "TARGET",
    "bureau_total_credit",
    "previous_total_credit",
    "historical_total_due",
    "bureau_debt_to_credit_ratio_clean",
    "bureau_overdue_to_credit_ratio_clean",
    "historical_payment_completion_ratio_clean",
    "historical_overpayment_ratio",
    "historical_overpayment_flag"
]

missing_required_columns = [
    column
    for column in required_columns
    if column not in hc_silver_final.columns
]


# ============================================================
# Clean ratio checks
# ============================================================

ratio_features = [
    "bureau_debt_to_credit_ratio_clean",
    "bureau_overdue_to_credit_ratio_clean",
    "historical_payment_completion_ratio_clean"
]

invalid_ratio_count = 0

for feature in ratio_features:

    invalid_count = (
        hc_silver_final
        .filter(
            (F.col(feature) < 0) |
            (F.col(feature) > 1)
        )
        .count()
    )

    invalid_ratio_count += invalid_count


# ============================================================
# Clean debt checks
# ============================================================

negative_debt_count = (
    hc_silver_final
    .filter(
        F.col("bureau_total_debt_clean") < 0
    )
    .count()
)

negative_max_debt_count = (
    hc_silver_final
    .filter(
        F.col("bureau_max_debt_clean") < 0
    )
    .count()
)


# ============================================================
# Quality checks
# ============================================================

quality_checks = [
    (
        "Expected application rows = 307,511",
        row_count == 307511
    ),
    (
        "One row per SK_ID_CURR",
        duplicate_customer_ids == 0
    ),
    (
        "Distinct SK_ID_CURR = 307,511",
        distinct_customer_ids == 307511
    ),
    (
        "TARGET column present",
        target_present
    ),
    (
        "All required engineered features present",
        len(missing_required_columns) == 0
    ),
    (
        "Cleaned ratios within [0, 1]",
        invalid_ratio_count == 0
    ),
    (
        "No negative cleaned bureau debt",
        negative_debt_count == 0
    ),
    (
        "No negative cleaned maximum bureau debt",
        negative_max_debt_count == 0
    ),
    (
        "Expected cleaned Silver columns = 180",
        column_count == 180
    )
]


# ============================================================
# Display results
# ============================================================

quality_gate_df = spark.createDataFrame(
    quality_checks,
    [
        "quality_check",
        "passed"
    ]
)

display(
    quality_gate_df
)


# ============================================================
# Final result
# ============================================================

all_checks_passed = all(
    check[1]
    for check in quality_checks
)

print("\n" + "=" * 80)

if all_checks_passed:
    print("FINAL HOME CREDIT SILVER QUALITY GATE: PASSED")
else:
    print("FINAL HOME CREDIT SILVER QUALITY GATE: REVIEW REQUIRED")

print("=" * 80)

if missing_required_columns:
    print(
        f"Missing required columns: "
        f"{missing_required_columns}"
    )

FINAL HOME CREDIT SILVER QUALITY GATE


quality_check,passed
"Expected application rows = 307,511",true
One row per SK_ID_CURR,true
"Distinct SK_ID_CURR = 307,511",true
TARGET column present,true
All required engineered features present,true
"Cleaned ratios within [0, 1]",true
No negative cleaned bureau debt,true
No negative cleaned maximum bureau debt,true
Expected cleaned Silver columns = 180,true



FINAL HOME CREDIT SILVER QUALITY GATE: PASSED


## Final Home Credit Silver Quality Gate

The Home Credit engineering pipeline successfully produces a cleaned Silver dataset at the application level. The final quality gate verifies that the expected application population and customer-level grain are preserved, the target and required engineered historical features are present, and the robustly treated ratio and debt features satisfy their defined constraints.

The cleaned Silver layer retains the original source-derived features while providing robust versions of selected historical credit and payment features for downstream modelling. The original Bronze data remains unchanged, preserving a clear separation between source data and modelling-oriented treatment.

This notebook therefore completes the Home Credit Bronze-to-Silver engineering stage. Gold-layer feature selection and modelling preparation are performed in the subsequent notebook.